<a href="https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Flag pages that rank well (good average position) but have surprisingly low CTR for that position — these are "CTR-fix" opportunities, prioritized by how much traffic volume they carry (bigger volume = bigger payoff from fixing the title/snippet).

Reason codes: CTR_FIX_OPPORTUNITY (good position, underperforming CTR), MONITOR (no clear signal).

Signal 1 checked: CTR-vs-position (this is the real signal behind FlyRank's CTR-fix logic from this week's session).
Signal 2 checked: volume/impressions (behind the quick-win logic — bigger volume means a fix has bigger absolute payoff).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Signal 1: CTR vs position — bucket by position tier
sig1 = con.sql(f"""
    SELECT
        CASE WHEN gsc_sum_position/NULLIF(gsc_impressions,0) <= 3 THEN '1-3 (top)'
             WHEN gsc_sum_position/NULLIF(gsc_impressions,0) <= 10 THEN '4-10'
             WHEN gsc_sum_position/NULLIF(gsc_impressions,0) <= 20 THEN '11-20'
             ELSE '21+' END AS position_tier,
        AVG(gsc_clicks/NULLIF(gsc_impressions,0)) AS avg_ctr,
        COUNT(*) AS n
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()
print("Signal 1 — CTR by position tier:")
print(sig1)
print("Verdict: CONFIRMED — CTR clearly drops as position tier worsens, matching the assumption behind the CTR-fix flag.\n")

# Signal 2: volume (impressions) tiers
sig2 = con.sql(f"""
    SELECT
        CASE WHEN gsc_impressions < 100 THEN 'low (<100)'
             WHEN gsc_impressions < 1000 THEN 'medium (100-999)'
             ELSE 'high (1000+)' END AS volume_tier,
        AVG(gsc_clicks) AS avg_clicks,
        COUNT(*) AS n
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
    ORDER BY 2
""").df()
print("Signal 2 — clicks by volume tier:")
print(sig2)
print("Verdict: CONFIRMED — higher-volume pages generate proportionally more clicks, so fixing them has bigger absolute payoff (supports quick-win logic).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 — CTR by position tier:
  position_tier   avg_ctr        n
0     1-3 (top)  0.004756   727362
1         11-20  0.002770   519223
2           21+  0.001289   908354
3          4-10  0.003473  1456122
Verdict: CONFIRMED — CTR clearly drops as position tier worsens, matching the assumption behind the CTR-fix flag.

Signal 2 — clicks by volume tier:
        volume_tier  avg_clicks        n
0        low (<100)    0.057965  2972453
1  medium (100-999)    0.800425   606189
2      high (1000+)    5.068787    32419
Verdict: CONFIRMED — higher-volume pages generate proportionally more clicks, so fixing them has bigger absolute payoff (supports quick-win logic).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule encoding: score = predicted CTR-fix opportunity size = (position_tier_gap) x (impressions), where position_tier_gap is how much CTR is underperforming for that position tier. Action label: CTR_FIX_OPPORTUNITY if score is high, MONITOR otherwise. Reason code explains why.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Build per-page aggregates for the month
queue = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_sum_position / NULLIF(gsc_impressions,0)) AS avg_position,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING impressions >= 100
""").df()

# Expected CTR by position tier (from Signal 1 analysis)
def expected_ctr(pos):
    if pos <= 3: return 0.004756
    elif pos <= 10: return 0.003473
    elif pos <= 20: return 0.002770
    else: return 0.001289

queue['expected_ctr'] = queue['avg_position'].apply(expected_ctr)
queue['ctr_gap'] = queue['expected_ctr'] - queue['ctr']  # positive = underperforming
queue['score'] = queue['ctr_gap'].clip(lower=0) * queue['impressions']

queue['reason_code'] = 'MONITOR'
queue.loc[queue['ctr_gap'] > 0, 'reason_code'] = 'CTR_FIX_OPPORTUNITY'

queue['action_label'] = queue['reason_code']

queue = queue.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 101,441 ranked rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,expected_ctr,ctr_gap,score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.003473,0.003360,713.679092,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,0.004756,0.003336,678.831732,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,0.003473,0.003466,467.799432,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,0.003473,0.003172,453.704987,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,0.003473,0.003465,429.912475,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,0.003473,0.002847,377.495489,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,0.003473,0.003334,358.639232,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
7,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,0.004756,0.004323,349.384676,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
8,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,0.004756,0.001503,332.550360,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY
9,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,2.481596,0.001031,0.004756,0.003725,332.373124,CTR_FIX_OPPORTUNITY,CTR_FIX_OPPORTUNITY


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 rows from the ranked queue — for each: the action, why it's flagged, and what would make this pick wrong.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()
top20['why_flagged'] = top20.apply(
    lambda r: f"Position {r['avg_position']:.1f} but CTR only {r['ctr']:.4f} vs expected {r['expected_ctr']:.4f} ({r['impressions']:.0f} impressions at stake)",
    axis=1
)
top20['what_would_make_it_wrong'] = (
    "If the low CTR is actually due to a mismatched search intent (not a fixable title/snippet issue), "
    "or if this page's traffic is seasonal/one-off rather than a stable pattern, fixing the snippet won't help."
)

for i, row in top20.iterrows():
    print(f"#{i+1} | {row['action_label']} | {row['why_flagged']}")
    print(f"    Would be wrong if: {row['what_would_make_it_wrong']}\n")

#1 | CTR_FIX_OPPORTUNITY | Position 7.3 but CTR only 0.0001 vs expected 0.0035 (212404 impressions at stake)
    Would be wrong if: If the low CTR is actually due to a mismatched search intent (not a fixable title/snippet issue), or if this page's traffic is seasonal/one-off rather than a stable pattern, fixing the snippet won't help.

#2 | CTR_FIX_OPPORTUNITY | Position 2.6 but CTR only 0.0014 vs expected 0.0048 (203497 impressions at stake)
    Would be wrong if: If the low CTR is actually due to a mismatched search intent (not a fixable title/snippet issue), or if this page's traffic is seasonal/one-off rather than a stable pattern, fixing the snippet won't help.

#3 | CTR_FIX_OPPORTUNITY | Position 4.5 but CTR only 0.0000 vs expected 0.0035 (134984 impressions at stake)
    Would be wrong if: If the low CTR is actually due to a mismatched search intent (not a fixable title/snippet issue), or if this page's traffic is seasonal/one-off rather than a stable pattern, fixing the snippet

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick identification: rows with CTR near exactly 0.0000 despite high impressions could be a genuine intent-mismatch problem (not fixable by title/snippet), not real CTR-fix opportunities — worth flagging as lower-confidence picks. Leakage check: confirmed no future-window or label-derived columns were used — all features (impressions, clicks, position, CTR) come only from the same month's own observable GSC data, no product flags, no forward-looking labels.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag potentially weak picks: near-zero CTR might mean intent mismatch, not a fixable snippet
weak_picks = queue.head(20)[queue.head(20)['ctr'] < 0.0002]
print(f"Weak/lower-confidence picks in top 20: {len(weak_picks)}")
print(weak_picks[['client_hash_id','content_hash_id','impressions','ctr','avg_position']].to_string())

print("\n--- Leakage check ---")
features_used = ['impressions','clicks','avg_position','ctr','expected_ctr','ctr_gap']
print("Features used in scoring:", features_used)
print("All features are computed only from the current month's own gsc_impressions/gsc_clicks/gsc_sum_position.")
print("No product flags, no future-window columns, no label-derived inputs were used.")

Weak/lower-confidence picks in top 20: 7
             client_hash_id           content_hash_id  impressions       ctr  avg_position
0   client_23a62021009f63c4  content_44f34c0a90047651     212404.0  0.000113      7.346909
2   client_73cda7b4e4f265ea  content_8e1334d6356668e3     134984.0  0.000007      4.545582
4   client_73cda7b4e4f265ea  content_fec55986a1868d62     124075.0  0.000008      9.385150
6   client_62f4a7e64f5e0096  content_f6116743b00afc2d     107584.0  0.000139      9.536301
12  client_9958f0a7ae1df715  content_cd3d932d4e1c8db0      89332.0  0.000045      7.786219
17  client_a80fca3f171ed1de  content_046fc480045b88f5      83788.0  0.000072      7.289152
18  client_a80fca3f171ed1de  content_9540d884af3e41fd      82376.0  0.000134      7.794395

--- Leakage check ---
Features used in scoring: ['impressions', 'clicks', 'avg_position', 'ctr', 'expected_ctr', 'ctr_gap']
All features are computed only from the current month's own gsc_impressions/gsc_clicks/gsc_sum_position.
N

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.